# RAG Systematic Evaluation Pipeline — Interactive Tour

This notebook walks through every layer of the pipeline: how documents get chunked, how chunks become searchable embeddings, how three retrieval strategies work, how we measure quality, and what the 24-experiment grid search reveals.

## What You Will Learn

- **Chunking strategies** — fixed-size, sentence-based, and semantic splitting, and why the choice matters for downstream retrieval quality
- **Dense retrieval** — how text is mapped to vectors and why similarity search finds semantically related content without exact keyword overlap
- **BM25 and hybrid retrieval** — how sparse keyword matching works and how fusing dense + sparse scores with an alpha parameter can improve recall
- **IR evaluation metrics** — MRR, Recall@K, and NDCG@K explained step by step with worked numeric examples
- **24-experiment grid results** — loading and interpreting the pre-computed experiment grid, ranking configurations, and reading pivot tables
- **Visualizations** — using the project's built-in chart generator to inspect performance across all dimensions

## Prerequisites

- Python 3.12
- `rag_common` package installed (`uv pip install -e ../rag-common` from the project root)
- `numpy`, `pandas`, `matplotlib`, `seaborn` — included in the project dependencies
- Sections 3–6 run fully offline using stub embeddings — **no API key required**
- Sections 7–8 load pre-existing artifacts from `../experiments/` and `../visualizations/`

---
## Section 1 — Setup & Imports

In [ ]:
import json
import sys
from pathlib import Path

# Make the project root importable so src.config and src.visualizer resolve.
# The notebook lives in notebooks/, so we go one level up.
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib
import numpy as np
import pandas as pd
import seaborn as sns

# rag_common chunkers — the shared library used by the pipeline
from rag_common.chunkers import FixedSizeChunker, SentenceBasedChunker

matplotlib.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
# print("Project root:", PROJECT_ROOT)
print("Experiments directory exists:", (PROJECT_ROOT / "experiments").exists())

---
## Section 2 — Chunking Strategies

Before a document can be retrieved, it must be split into smaller passages called **chunks**. The chunking strategy is one of the most consequential decisions in a RAG pipeline:

- **Too large**: chunks contain multiple topics, so the retrieved passage may include irrelevant content that dilutes the answer.
- **Too small**: answers may be split across chunk boundaries and the retriever misses the complete context.
- **Wrong boundary**: splitting a sentence in the middle produces awkward, incomplete passages.

This pipeline tests three chunking strategies across four configurations:

| Strategy | Key parameter | Description |
|---|---|---|
| `fixed_size` | `chunk_size` (chars) | Slide a character window; snap to word boundaries |
| `sentence` | `sentences_per_chunk` | Group N sentences; carry over `overlap_sentences` |
| `semantic` | `breakpoint_threshold` | Split where cosine similarity between adjacent sentences drops below threshold |

Below we demonstrate the two simpler strategies on a short paragraph. `SemanticChunker` requires an embedding function, so we introduce it later in Section 3 alongside the stub embedder.

In [ ]:
# A 10-sentence paragraph on retrieval-augmented generation
DEMO_TEXT = (
    "Retrieval-augmented generation (RAG) is a technique that combines a large language model "
    "with an external knowledge source. "
    "Rather than relying solely on parametric knowledge baked into model weights, RAG retrieves "
    "relevant passages at inference time. "
    "This allows the model to ground its responses in up-to-date or domain-specific information. "
    "The retrieval step typically uses either sparse keyword matching or dense embedding similarity. "
    "Sparse methods like BM25 excel at exact keyword matches and are computationally cheap. "
    "Dense methods use neural embeddings to capture semantic meaning beyond surface keywords. "
    "Hybrid approaches fuse both scores to balance precision and recall. "
    "The quality of retrieved passages depends heavily on how the source document is chunked. "
    "Poor chunking can cause answers to be split across passage boundaries. "
    "Systematic evaluation over a grid of chunking and retrieval configurations reveals the optimal setup."
)

# --- Fixed-size chunking: 256-character window, 50-character overlap ---
fixed_chunker = FixedSizeChunker(chunk_size=256, overlap=50)
fixed_chunks = fixed_chunker.chunk(DEMO_TEXT)

# --- Sentence-based chunking: 3 sentences per chunk, 1-sentence overlap ---
sentence_chunker = SentenceBasedChunker(sentences_per_chunk=3, overlap_sentences=1)
sentence_chunks = sentence_chunker.chunk(DEMO_TEXT)

print(f"Fixed-size  => {len(fixed_chunks)} chunks")
print(f"Sentence    => {len(sentence_chunks)} chunks")
print()

print("=== Fixed-size: first chunk ===")
print(repr(fixed_chunks[0].content))
print()
print("=== Sentence-based: first chunk ===")
print(repr(sentence_chunks[0].content))

In [ ]:
# Build a comparison DataFrame
def chunk_stats(chunks, label):
    lengths = [len(c.content) for c in chunks]
    return {
        "strategy": label,
        "num_chunks": len(chunks),
        "avg_chars": round(np.mean(lengths), 1),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
    }


comparison_df = pd.DataFrame(
    [
        chunk_stats(fixed_chunks, "fixed_size(256, ol=50)"),
        chunk_stats(sentence_chunks, "sentence(3 sents, ol=1)"),
    ]
)
comparison_df.set_index("strategy", inplace=True)
comparison_df

**Takeaway:** Fixed-size chunking is fast and parameter-simple, but it cuts text at character boundaries and can split a sentence in the middle. Sentence-based chunking respects natural language boundaries and makes each chunk semantically self-contained, which usually improves retrieval precision.

---
## Section 3 — Embedding & Dense Retrieval

**Embedding** converts text into a dense numerical vector. Similar texts end up with similar vectors, so we can find relevant passages by computing cosine similarity between the query vector and all chunk vectors.

For this tutorial we use a **deterministic stub embedder** that requires no API key: for any given text, it derives a fixed random seed from the first 40 characters, then generates a unit vector from that seed. The same text always produces the same vector — the vectors are not *semantically* meaningful, but the mechanics of FAISS indexing and similarity search are identical.

### How dense retrieval works

1. **Index time**: embed every chunk and store vectors in a vector index (FAISS, Pinecone, etc.).
2. **Query time**: embed the query and find the nearest vectors by cosine or dot-product similarity.
3. Return the top-K chunks.

In [ ]:
# Stub embedder: deterministic hash-seeded unit vectors (no API key needed)
DIM = 64


def stub_embed(texts: list) -> np.ndarray:
    """Return (N, DIM) float32 unit vectors derived deterministically from text."""
    vecs = []
    for t in texts:
        seed = abs(hash(t[:40])) % (2**31)
        v = np.random.default_rng(seed).standard_normal(DIM).astype(np.float32)
        vecs.append(v / np.linalg.norm(v))
    return np.array(vecs)


# Verify: same text always produces the same vector
v1 = stub_embed(["hello world"])[0]
v2 = stub_embed(["hello world"])[0]
assert np.allclose(v1, v2), "Stub embedder must be deterministic"
print("Stub embedder verified: deterministic and unit-norm")
print(f"Output shape for 3 texts: {stub_embed(['a', 'b', 'c']).shape}")

In [ ]:
# Build a mini corpus of 5 short chunks about RAG topics
mini_corpus = [
    "BM25 is a classic sparse retrieval algorithm based on term frequency and inverse document frequency.",
    "Dense retrieval uses neural embeddings to find semantically similar documents.",
    "Chunking splits documents into smaller passages before indexing them.",
    "Hybrid retrieval combines sparse and dense scores using a weighted sum controlled by alpha.",
    "NDCG measures retrieval quality by rewarding relevant documents ranked higher in the result list.",
]

# Embed the corpus
corpus_vecs = stub_embed(mini_corpus)  # shape: (5, 64)
print(f"Corpus matrix shape: {corpus_vecs.shape}")
print(f"All unit vectors: {np.allclose(np.linalg.norm(corpus_vecs, axis=1), 1.0)}")

In [ ]:
# Run a dense retrieval query against the mini corpus.
# We use a direct matrix multiply (cosine similarity = dot product for unit vectors),
# which avoids needing FAISS installed in the tutorial environment.
query = "How does semantic similarity search work with embeddings?"
query_vec = stub_embed([query])[0]  # shape: (64,)

scores = corpus_vecs @ query_vec  # shape: (5,)
top3_idx = np.argsort(scores)[::-1][:3]

print(f"Query: '{query}'\n")
print(f"{'Rank':<6} {'Score':>8}  Chunk text")
print("-" * 80)
for rank, idx in enumerate(top3_idx, start=1):
    print(f"  {rank:<4} {scores[idx]:>8.4f}  {mini_corpus[idx]}")

**Takeaway:** The stub embedder used here produces hash-seeded random unit vectors — the ranking above is essentially random and does not demonstrate semantic similarity. With a real embedding model (`text-embedding-3-small` or `text-embedding-3-large`), semantically similar chunks cluster together in vector space and the query "How does semantic similarity search work with embeddings?" would rank the dense-retrieval chunk first by a meaningful margin. The quality of the embedding model determines how well semantic similarity maps to actual relevance — which is why the main project grid tests both `text-embedding-3-small` and `text-embedding-3-large`.

---
## Section 4 — BM25 & Hybrid Retrieval

**BM25** (Best Match 25) is the gold-standard sparse retrieval algorithm. It scores each document based on:

- **Term frequency (TF)**: how often does the query term appear in the chunk? Repeated terms add diminishing returns (saturation).
- **Inverse document frequency (IDF)**: how rare is the term across the corpus? Rare terms are more informative.
- **Document length normalisation**: longer chunks are penalised to prevent length bias.

BM25 is fast (no neural inference), interpretable, and works well when the query contains distinctive keywords present verbatim in the relevant chunk.

**Hybrid retrieval** fuses BM25 and dense scores:

```
final_score = alpha * dense_norm + (1 - alpha) * bm25_norm
```

Both score sets are min-max normalised to `[0, 1]` before fusion so that the raw magnitude of BM25 scores (unbounded positive floats) does not dominate cosine similarities (bounded in `[-1, 1]`).

In [ ]:
# --- BM25 retrieval on the same 5-chunk mini corpus ---
from rank_bm25 import BM25Okapi

tokenised_corpus = [doc.lower().split() for doc in mini_corpus]
bm25 = BM25Okapi(tokenised_corpus)

bm25_query = "sparse retrieval term frequency keyword matching"
bm25_scores_raw = bm25.get_scores(bm25_query.lower().split())

print(f"Query: '{bm25_query}'\n")
print(f"{'Rank':<6} {'BM25 Score':>12}  Chunk text")
print("-" * 80)
bm25_ranked = np.argsort(bm25_scores_raw)[::-1]
for rank, idx in enumerate(bm25_ranked, start=1):
    print(f"  {rank:<4} {bm25_scores_raw[idx]:>12.4f}  {mini_corpus[idx]}")

In [ ]:
# --- Hybrid fusion (alpha=0.5) using the same query ---
hybrid_query = "sparse retrieval term frequency keyword matching"

q_vec = stub_embed([hybrid_query])[0]
dense_scores_raw = corpus_vecs @ q_vec


def minmax_norm(arr):
    lo, hi = arr.min(), arr.max()
    span = hi - lo
    return arr if span == 0 else (arr - lo) / span


alpha = 0.5
bm25_norm_arr = minmax_norm(bm25_scores_raw)
dense_norm_arr = minmax_norm(dense_scores_raw)
hybrid_scores = alpha * dense_norm_arr + (1 - alpha) * bm25_norm_arr

# Side-by-side comparison table
fusion_df = pd.DataFrame(
    {
        "chunk": [f"chunk_{i}" for i in range(len(mini_corpus))],
        "bm25_norm": bm25_norm_arr.round(4),
        "dense_norm": dense_norm_arr.round(4),
        "hybrid": hybrid_scores.round(4),
        "text": [t[:55] + "..." for t in mini_corpus],
    }
)
fusion_df.sort_values("hybrid", ascending=False, inplace=True)
fusion_df.reset_index(drop=True, inplace=True)
fusion_df.index = range(1, len(fusion_df) + 1)
fusion_df.index.name = "hybrid_rank"
fusion_df

**Takeaway:** Hybrid retrieval combines the precision of dense semantic search with the keyword-recall of BM25. The `alpha` parameter controls the balance: `alpha=1.0` is pure dense, `alpha=0.0` is pure BM25. On government-document corpora with precise but repeated terminology (like `fy10syb.pdf`), BM25 can introduce noise — which is why the grid results show that pure vector retrieval often outperforms hybrid at `alpha=0.5`.

---
## Section 5 — Evaluation Metrics

To compare retrieval configurations objectively we need metrics that quantify how well the retrieved list matches the ground truth. The P3 pipeline uses three standard information retrieval metrics.

### Mean Reciprocal Rank (MRR)

For each query, find the rank position of the **first** relevant result:

```
RR  = 1 / rank_of_first_relevant_result   (0 if not found)
MRR = mean(RR) over all queries
```

MRR is the primary metric when users care about getting **at least one** correct answer quickly.

### Recall@K

```
Recall@K = |relevant ∩ top-K| / |relevant|
```

Measures what fraction of all relevant chunks were retrieved in the top K. Important when downstream LLM generation needs comprehensive coverage.

### NDCG@K (Normalised Discounted Cumulative Gain)

```
DCG@K  = Σ rel_i / log2(i + 1)   for i in 1..K   (binary relevance)
IDCG@K = DCG of the ideal (perfect) ranking
NDCG@K = DCG@K / IDCG@K
```

Higher-ranked relevant results contribute more to the score (logarithmic discount). Bounded in `[0, 1]`. Best metric when both relevance and rank position matter.

In [ ]:
import math

# Worked manual example:
# One relevant chunk (chunk_B), three retrieval systems with different rank positions.

relevant = {"chunk_B"}

system_a = ["chunk_A", "chunk_B", "chunk_C", "chunk_D", "chunk_E"]  # relevant at rank 2
system_b = ["chunk_B", "chunk_A", "chunk_C", "chunk_D", "chunk_E"]  # relevant at rank 1
system_c = ["chunk_C", "chunk_D", "chunk_E", "chunk_A", "chunk_B"]  # relevant at rank 5

K = 5


def rr(retrieved, rel):
    for rank, rid in enumerate(retrieved, start=1):
        if rid in rel:
            return 1.0 / rank
    return 0.0


def recall_at_k(retrieved, rel, k):
    return len(set(retrieved[:k]) & rel) / len(rel)


def ndcg_at_k(retrieved, rel, k):
    dcg = sum(
        1.0 / math.log2(rank + 1) for rank, rid in enumerate(retrieved[:k], start=1) if rid in rel
    )
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


rows = []
for name, system in [
    ("System A (relevant @rank 2)", system_a),
    ("System B (relevant @rank 1)", system_b),
    ("System C (relevant @rank 5)", system_c),
]:
    rows.append(
        {
            "system": name,
            "RR": round(rr(system, relevant), 4),
            f"Recall@{K}": round(recall_at_k(system, relevant, K), 4),
            f"NDCG@{K}": round(ndcg_at_k(system, relevant, K), 4),
        }
    )

metrics_df = pd.DataFrame(rows).set_index("system")
metrics_df

In [ ]:
# Show how rag_common.metrics.evaluate() wraps all of this into one call
from rag_common.metrics import evaluate

# evaluate() expects a list of (retrieved_ids, relevant_ids_set) tuples
query_results = [
    (system_a, relevant),
    (system_b, relevant),
    (system_c, relevant),
]

scores = evaluate(query_results, k=5)
print("evaluate() output (aggregated over 3 simulated queries):")
for metric, value in scores.items():
    print(f"  {metric}: {value:.4f}")

**Takeaway:** Use **MRR** when you need the first correct answer to appear as high as possible — this is the primary P3 metric. Use **Recall@5** when your LLM generator needs full coverage of relevant content (e.g., multi-hop questions). Use **NDCG@5** when you want a single score that rewards both rank quality and coverage. In P3, all three tend to agree because each query has exactly one relevant chunk, but the differences become meaningful when relevant chunks number more than one.

---
## Section 6 — 24-Experiment Grid Results

The evaluation grid covers:

- **4 chunking configs**: `fixed_256_ol50`, `fixed_512_ol100`, `sentence_5s_ol1`, `semantic_t0.65_max10`
- **2 embedding models**: `text-embedding-3-small` (1536-dim), `text-embedding-3-large` (3072-dim)
- **3 retrieval methods**: `vector`, `bm25`, `hybrid_a0.5`

= **24 experiments**, each evaluated on 25 synthetic QA pairs generated from `data/fy10syb.pdf`.

Results are stored as JSON files in `../experiments/`. We load them all, flatten into a DataFrame, and rank by MRR.

In [ ]:
EXPERIMENTS_DIR = Path("../experiments")
experiment_files = sorted(EXPERIMENTS_DIR.glob("*.json"))
print(f"Found {len(experiment_files)} experiment files")

raw_dicts = []
for fp in experiment_files:
    with open(fp) as f:
        raw_dicts.append(json.load(f))

print(f"Loaded {len(raw_dicts)} experiment results")
print("Top-level keys in each result:", list(raw_dicts[0].keys()))

In [ ]:
# Parse each result into a flat DataFrame row
def parse_result(d: dict) -> dict:
    cfg = d["config"]
    m = d["metrics"]
    return {
        "experiment_id": d["experiment_id"],
        "chunk_strategy": cfg["chunk"]["strategy"],
        "embed_model": cfg["embed"]["model"].split("-")[-1],  # 'small' or 'large'
        "retrieval_method": cfg["retrieval"]["method"],
        "mrr": m["mrr"],
        "ndcg@5": m["ndcg_at_k"].get("5", 0.0),
        "recall@5": m["recall_at_k"].get("5", 0.0),
        "avg_latency_s": m.get("avg_retrieval_time_s", 0.0),
    }


df_results = pd.DataFrame([parse_result(d) for d in raw_dicts])
df_results.sort_values("mrr", ascending=False, inplace=True)
df_results.reset_index(drop=True, inplace=True)
df_results.index = range(1, len(df_results) + 1)
df_results.index.name = "rank"

print("=== Top 10 Configurations by MRR ===")
df_results[
    [
        "experiment_id",
        "chunk_strategy",
        "embed_model",
        "retrieval_method",
        "mrr",
        "ndcg@5",
        "recall@5",
        "avg_latency_s",
    ]
].head(10)

In [ ]:
# Pivot table: chunk_strategy x retrieval_method, values = mean MRR
# (averaged over both embedding models)
pivot = df_results.pivot_table(
    index="chunk_strategy",
    columns="retrieval_method",
    values="mrr",
    aggfunc="mean",
).round(4)

# Reorder columns so dense/hybrid/sparse appear left-to-right
col_order = [c for c in ["vector", "hybrid", "bm25"] if c in pivot.columns]
pivot = pivot[col_order]

print("=== Mean MRR: Chunk Strategy x Retrieval Method (avg over embed models) ===")
pivot

**Takeaway:** The pivot table isolates the contribution of each axis. `semantic` chunking consistently outperforms fixed-size strategies, and `vector` retrieval beats `bm25` on this corpus. The grid structure is what makes these conclusions robust — each cell is the average of multiple embedding model runs, so the signal is not model-specific noise.

---
## Section 7 — Visualizations

The project's `src.visualizer.generate_all_charts()` function produces 9 charts covering the full evaluation surface:

1. **MRR leaderboard** — all 24 configs ranked by MRR
2. **Recall@K curves** — mean Recall@K for K in {1,3,5,10} by retrieval method
3. **Chunking comparison** — grouped bars: chunk strategy × core metrics
4. **Embedding model comparison** — small vs large across MRR, MAP, Recall@5, NDCG@5
5. **Retrieval method comparison** — vector / BM25 / hybrid side by side
6. **MRR heatmap** — chunk config × embedding model
7. **Recall@5 vs Precision@5 scatter** — with top-5 configs labelled
8. **Metric correlation matrix** — Pearson r across all IR metrics
9. **Response time vs quality** — latency vs MRR with Pareto front annotated

The function takes `list[EvaluationResult]` (Pydantic models). We reconstruct these from the JSON dicts already loaded — no re-running the experiments.

In [ ]:
from IPython.display import Image, display

from src.config import EvaluationResult
from src.visualizer import generate_all_charts

# Reconstruct Pydantic EvaluationResult objects from the loaded JSON dicts.
# model_validate() handles enum coercion (e.g. 'vector' -> RetrievalMethod.VECTOR)
# and nested model validation automatically.
eval_results = [EvaluationResult.model_validate(d) for d in raw_dicts]
print(f"Reconstructed {len(eval_results)} EvaluationResult objects")

# Spot-check the best result
best = max(eval_results, key=lambda r: r.metrics.mrr)
print(f"Best config: {best.experiment_id}  (MRR={best.metrics.mrr:.4f})")

In [ ]:
# Generate (or regenerate) all 9 charts.
# Charts are saved to ../visualizations/ as PNG files.
VIZ_DIR = Path("../visualizations")

chart_paths = generate_all_charts(eval_results, output_dir=VIZ_DIR)

print("Charts saved:")
for name, path in chart_paths.items():
    print(f"  {name}: {path.name}")

In [ ]:
# Display the MRR leaderboard inline
leaderboard_path = chart_paths.get("mrr_leaderboard", VIZ_DIR / "mrr_leaderboard.png")
if leaderboard_path.exists():
    display(Image(filename=str(leaderboard_path)))
else:
    print(f"Chart not found at {leaderboard_path}")

In [ ]:
# Display the MRR heatmap (chunk config x embedding model)
heatmap_path = chart_paths.get("mrr_heatmap", VIZ_DIR / "mrr_heatmap.png")
if heatmap_path.exists():
    display(Image(filename=str(heatmap_path)))
else:
    print(f"Chart not found at {heatmap_path}")

In [ ]:
# Display the remaining 7 charts
display_order = [
    "recall_at_k_curves",
    "chunking_comparison",
    "embedding_comparison",
    "retrieval_comparison",
    "recall_precision_scatter",
    "metric_correlation",
    "response_time_vs_quality",
]

for chart_name in display_order:
    path = chart_paths.get(chart_name, VIZ_DIR / f"{chart_name}.png")
    if path.exists():
        print(f"--- {chart_name} ---")
        display(Image(filename=str(path)))
    else:
        print(f"Chart not found: {chart_name}")

---
## Section 8 — Key Findings & Takeaways

### Best Configuration

**`semantic_t0.65_max10__large__vector`** achieved the highest scores across the board:

- MRR = 0.9280
- Recall@5 = 1.0 (the relevant chunk appeared in every query's top-5 results)
- NDCG@5 = 0.9459

This configuration uses semantic chunking (threshold 0.65, max 10 sentences per chunk), the `text-embedding-3-large` model, and pure vector (dense) retrieval.

### Chunking: Semantic Beats Fixed-size on Dense Government Text

Government budget documents like `fy10syb.pdf` have consistent, information-dense sentence structure with frequent topic shifts between paragraphs. Semantic chunking detects these shifts via cosine similarity drops between sentence embeddings, creating chunks that are semantically coherent regardless of character count. Fixed-size chunking cuts at arbitrary character boundaries, frequently splitting a sentence or fusing two unrelated topics into one chunk — both of which hurt retrieval precision.

### Vector Retrieval Outperforms Hybrid at alpha=0.5

Hybrid retrieval at `alpha=0.5` performed below pure vector retrieval in most configurations. The root cause is pool contamination during min-max normalisation. BM25 IDF *penalises* frequent terms (budget category names and section headers score low), but the introduction chunk — which concentrates many rare technical terms — scores 8–14× higher than the next candidate. After min-max normalisation this outlier locks in at score=1.0 and compresses all other BM25 scores toward zero, making the `alpha` weight effectively irrelevant in practice. Pure vector retrieval is unaffected because cosine similarities are bounded in [−1, 1] and do not exhibit this outlier distortion. Rank-based fusion (RRF) would eliminate the problem entirely by discarding raw scores.

### Embedding Model: Mixed Results, Not a Uniform Win for Large

The result depends on chunking strategy. `text-embedding-3-large` wins on semantic chunking (+0.018 MRR, 0.928 vs 0.910) and fixed_512 (+0.130 MRR). But `text-embedding-3-small` wins on sentence chunking (+0.099 MRR, 0.860 vs 0.761 for vector retrieval) and fixed_256 (+0.015 MRR). The marginal gain on semantic chunking does not justify the 3× embedding cost increase for this corpus size and query distribution — `semantic + small + vector` at MRR=0.910 is the better cost-quality tradeoff.

### Latency vs Quality Trade-off

Semantic chunking with the large model and vector retrieval is the slowest configuration (dense embedding is slower than BM25, and semantic chunking itself requires an embedding pass during indexing). For latency-sensitive applications:

- `sentence_5s_ol1__small__vector` is the best latency-quality tradeoff: MRR=0.860 (92.7% of best) at 207ms — 28% faster than the winning configuration (287ms). Note that `sentence_5s_ol1__large__vector` is actually *slower* (302ms) than the best configuration while delivering only 82% of its MRR, making it a poor tradeoff on both axes.
- `fixed_512_ol100__large__vector` is the fastest dense configuration with still-acceptable quality.

### What to Try Next

- **Alpha sweep**: test hybrid at `alpha` values of 0.2, 0.3, 0.7, 0.8 to find the optimal BM25/dense balance for this corpus.
- **Reranking**: add a cross-encoder reranker (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`) after initial retrieval to push the best chunk to rank 1.
- **Over-retrieve then rerank**: retrieve top-20 with BM25 for high recall, then rerank to top-5 for precision.
- **Semantic chunking threshold sweep**: test thresholds of 0.5, 0.6, 0.7, 0.75 to see how chunk granularity affects the quality/latency trade-off on this document.